In [ ]:
# Example 1: Basic Memory Statistics and Object Counting
# Demonstrate objgraph's object counting and memory statistics

import objgraph
import gc

def create_sample_objects():
    """Create various objects to demonstrate objgraph functionality"""
    # Create some objects to analyze
    sample_list = [1, 2, 3, "hello", "world"]
    sample_dict = {"key1": "value1", "key2": [1, 2, 3]}
    sample_tuple = (1, "tuple", {"nested": "dict"})
    
    class SampleClass:
        def __init__(self, name):
            self.name = name
            self.data = [i for i in range(10)]
    
    objects = [SampleClass(f"object_{i}") for i in range(5)]
    
    return sample_list, sample_dict, sample_tuple, objects

# Force garbage collection to get cleaner stats
gc.collect()

print("=== Initial Memory Statistics ===")
print(f"Most common objects (top 10):")
for obj_type, count in objgraph.most_common_types(10):
    print(f"  {obj_type}: {count}")

print(f"\nTotal objects being tracked: {len(gc.get_objects())}")

# Create sample objects and see the change
print("\n=== After Creating Sample Objects ===")
sample_data = create_sample_objects()

# Show growth in specific object types
print(f"List objects: {objgraph.count('list')}")
print(f"Dict objects: {objgraph.count('dict')}")
print(f"SampleClass objects: {objgraph.count('SampleClass')}")
assert objgraph.count('dict') > 0
assert objgraph.count('SampleClass') == 5  # the five instances held by sample_data

# Show object growth since a reference point
objgraph.show_growth(limit=5)

print("Example 1: Basic memory statistics completed successfully!")

In [ ]:
# Example 2: Object Reference Tracking and Leak Detection
# Demonstrate reference tracking and memory leak detection

import objgraph
import gc
import weakref

class LeakyClass:
    """Class that can create circular references"""
    instances = []
    
    def __init__(self, name):
        self.name = name
        self.references = []
        LeakyClass.instances.append(self)
    
    def add_reference(self, other):
        """Add reference to another object"""
        self.references.append(other)
        if hasattr(other, 'references'):
            other.references.append(self)  # Create circular reference

def demonstrate_reference_tracking():
    """Show how to track object references"""
    
    # Create objects with circular references
    print("=== Creating Objects with Circular References ===")
    obj1 = LeakyClass("Object1")
    obj2 = LeakyClass("Object2") 
    obj3 = LeakyClass("Object3")
    
    # Create circular references
    obj1.add_reference(obj2)
    obj2.add_reference(obj3)
    obj3.add_reference(obj1)
    
    print(f"LeakyClass instances before cleanup: {objgraph.count('LeakyClass')}")
    
    # Show references to a specific object
    print(f"\nReferences to obj1:")
    refs = objgraph.find_backref_chain(obj1, objgraph.is_proper_module)[:3]
    for i, ref in enumerate(refs):
        print(f"  {i}: {type(ref)} - {getattr(ref, 'name', str(ref)[:50])}")
    
    return obj1, obj2, obj3

# Track objects before creating leaky references
initial_count = objgraph.count('LeakyClass')

# Create leaky objects
obj1, obj2, obj3 = demonstrate_reference_tracking()

print(f"\n=== Memory Cleanup Testing ===")
print(f"LeakyClass count before cleanup: {objgraph.count('LeakyClass')}")

# Try to clean up - this won't work due to circular references
del obj1, obj2, obj3
gc.collect()

print(f"LeakyClass count after del and gc.collect(): {objgraph.count('LeakyClass')}")

# Show how to find and break circular references
if objgraph.count('LeakyClass') > initial_count:
    print("Circular references detected - demonstrating cleanup")
    LeakyClass.instances.clear()  # Break the class-level reference
    gc.collect()
    print(f"LeakyClass count after breaking references: {objgraph.count('LeakyClass')}")

print("Example 2: Reference tracking and leak detection completed successfully!")

In [ ]:
# Example 3: Memory Growth Monitoring and Analysis
# Demonstrate continuous memory monitoring and growth analysis

import objgraph
import gc
import time

class DataProcessor:
    """Class to simulate memory-intensive operations"""
    def __init__(self):
        self.processed_data = []
        self.cache = {}
    
    def process_batch(self, batch_size=1000):
        """Process a batch of data that accumulates in memory"""
        batch = []
        for i in range(batch_size):
            item = {
                'id': i,
                'data': [j * 2 for j in range(50)],
                'metadata': {'timestamp': time.time(), 'processed': True}
            }
            batch.append(item)
            
            # Cache some items (potential memory growth)
            if i % 100 == 0:
                self.cache[f'cache_key_{i}'] = item.copy()
        
        self.processed_data.extend(batch)
        return len(batch)

def monitor_memory_growth(iterations=3):
    """Monitor memory growth over multiple iterations"""
    processor = DataProcessor()
    
    print("=== Memory Growth Monitoring ===")
    
    # Set baseline
    gc.collect()
    objgraph.show_most_common_types(limit=5)
    
    for iteration in range(iterations):
        print(f"\n--- Iteration {iteration + 1} ---")
        
        # Process data
        processed = processor.process_batch(500)
        
        # Show memory statistics
        print(f"Processed {processed} items")
        print(f"Total processed data: {len(processor.processed_data)}")
        print(f"Cache size: {len(processor.cache)}")
        
        # Show object counts for key types
        print(f"Dict objects: {objgraph.count('dict')}")
        print(f"List objects: {objgraph.count('list')}")
        print(f"DataProcessor objects: {objgraph.count('DataProcessor')}")
        
        # Show growth since last check
        if iteration > 0:
            print("Growth since last iteration:")
            objgraph.show_growth(limit=3)
    
    return processor

# Run memory growth monitoring
print("Starting memory growth analysis...")
processor = monitor_memory_growth(3)

print(f"\n=== Final Memory Analysis ===")
print(f"Final cache size: {len(processor.cache)}")
print(f"Final processed data count: {len(processor.processed_data)}")

# Demonstrate cleanup
print(f"\n=== Cleanup Demonstration ===")
initial_dict_count = objgraph.count('dict')
processor.cache.clear()
processor.processed_data.clear()
gc.collect()

final_dict_count = objgraph.count('dict')
print(f"Dict objects reduced by: {initial_dict_count - final_dict_count}")

print("Example 3: Memory growth monitoring completed successfully!")

In [ ]:
# Example 4: Object Graph Visualization and Reference Chains
# Demonstrate object graph visualization and reference chain analysis

import objgraph
import tempfile
import os

class Node:
    """Simple node class for creating object graphs"""
    def __init__(self, value, name=None):
        self.value = value
        self.name = name or f"Node_{value}"
        self.children = []
        self.parent = None
    
    def add_child(self, child):
        """Add a child node"""
        child.parent = self
        self.children.append(child)
    
    def __repr__(self):
        return f"Node({self.name}, children={len(self.children)})"

def create_object_graph():
    """Create a complex object graph for analysis"""
    # Create a tree-like structure
    root = Node(0, "Root")
    
    # Level 1
    child1 = Node(1, "Child1")
    child2 = Node(2, "Child2")
    root.add_child(child1)
    root.add_child(child2)
    
    # Level 2
    grandchild1 = Node(11, "GrandChild1")
    grandchild2 = Node(12, "GrandChild2")
    grandchild3 = Node(21, "GrandChild3")
    
    child1.add_child(grandchild1)
    child1.add_child(grandchild2)
    child2.add_child(grandchild3)
    
    # Create a circular reference for demonstration
    grandchild1.children.append(root)  # Circular reference!
    
    return root, [child1, child2], [grandchild1, grandchild2, grandchild3]

print("=== Object Graph Creation and Analysis ===")

# Create object graph
root, children, grandchildren = create_object_graph()

print(f"Created object graph:")
print(f"  Root node: {root}")
print(f"  Children: {children}")
print(f"  Grandchildren: {grandchildren}")

# Count our custom objects
print(f"\nNode objects in memory: {objgraph.count('Node')}")

# Find references to specific objects
print(f"\n=== Reference Analysis ===")

# Show what references our root object
root_refs = objgraph.find_backref_chain(root, objgraph.is_proper_module)
if root_refs:
    print(f"Reference chain to root (first 3 levels):")
    for i, ref in enumerate(root_refs[:3]):
        ref_info = f"{type(ref).__name__}"
        if hasattr(ref, 'name'):
            ref_info += f"({ref.name})"
        elif hasattr(ref, '__name__'):
            ref_info += f"({ref.__name__})"
        print(f"  Level {i}: {ref_info}")

# Analyze circular references
print(f"\n=== Circular Reference Detection ===")
print(f"Checking for circular references in our object graph...")

# Check if root is referenced by its descendants (indicating circular ref)
def has_circular_reference(node, visited=None):
    """Check if a node has circular references"""
    if visited is None:
        visited = set()
    
    if id(node) in visited:
        return True
    
    visited.add(id(node))
    
    for child in node.children:
        if isinstance(child, Node) and has_circular_reference(child, visited.copy()):
            return True
    
    return False

circular_detected = has_circular_reference(root)
print(f"Circular reference detected: {circular_detected}")

# Show object types and their relationships
print(f"\n=== Object Type Analysis ===")
print("Most common object types in our graph:")
for obj_type, count in objgraph.most_common_types(10):
    if obj_type in ['Node', 'list', 'dict', 'str']:
        print(f"  {obj_type}: {count}")

print("Example 4: Object graph visualization and analysis completed successfully!")

In [ ]:
# Example 5: Production Memory Profiling and Debugging Workflow
# Demonstrate comprehensive memory profiling for production debugging

import objgraph
import gc
import sys
from collections import defaultdict
import time

class MemoryProfiler:
    """Production-ready memory profiler using objgraph"""
    
    def __init__(self):
        self.snapshots = []
        self.baseline = None
        self.alerts = []
    
    def take_snapshot(self, label=""):
        """Take a memory snapshot"""
        gc.collect()  # Clean up first
        
        snapshot = {
            'label': label,
            'timestamp': time.time(),
            'total_objects': len(gc.get_objects()),
            'top_types': objgraph.most_common_types(10),
            'memory_usage': sys.getsizeof(gc.get_objects())
        }
        
        self.snapshots.append(snapshot)
        
        if self.baseline is None:
            self.baseline = snapshot
        
        return snapshot
    
    def analyze_growth(self, threshold=1000):
        """Analyze memory growth between snapshots"""
        if len(self.snapshots) < 2:
            return "Need at least 2 snapshots for growth analysis"
        
        current = self.snapshots[-1]
        previous = self.snapshots[-2]
        
        growth_analysis = {
            'object_growth': current['total_objects'] - previous['total_objects'],
            'time_elapsed': current['timestamp'] - previous['timestamp'],
            'type_changes': {}
        }
        
        # Compare object type counts
        prev_types = dict(previous['top_types'])
        curr_types = dict(current['top_types'])
        
        for obj_type in set(prev_types.keys()) | set(curr_types.keys()):
            prev_count = prev_types.get(obj_type, 0)
            curr_count = curr_types.get(obj_type, 0)
            change = curr_count - prev_count
            
            if abs(change) > threshold:
                growth_analysis['type_changes'][obj_type] = change
                if change > threshold:
                    self.alerts.append(f"High growth in {obj_type}: +{change}")
        
        return growth_analysis
    
    def get_memory_report(self):
        """Generate comprehensive memory report"""
        if not self.snapshots:
            return "No snapshots available"
        
        latest = self.snapshots[-1]
        
        report = [
            "=== Memory Profiling Report ===",
            f"Total snapshots taken: {len(self.snapshots)}",
            f"Latest snapshot: {latest['label']}",
            f"Total objects: {latest['total_objects']:,}",
            "",
            "Top object types:",
        ]
        
        for obj_type, count in latest['top_types']:
            report.append(f"  {obj_type}: {count:,}")
        
        if len(self.snapshots) > 1:
            growth = self.analyze_growth()
            report.extend([
                "",
                "Growth since last snapshot:",
                f"  Object count change: {growth['object_growth']:+,}",
                f"  Time elapsed: {growth['time_elapsed']:.2f}s"
            ])
            
            if growth['type_changes']:
                report.append("  Significant type changes:")
                for obj_type, change in growth['type_changes'].items():
                    report.append(f"    {obj_type}: {change:+,}")
        
        if self.alerts:
            report.extend([
                "",
                "Alerts:",
                *[f"  ⚠️  {alert}" for alert in self.alerts[-5:]]  # Show last 5 alerts
            ])
        
        return "\n".join(report)

# Simulate production application with memory issues
class ProductionApp:
    """Simulate a production application that may have memory issues"""
    
    def __init__(self):
        self.cache = {}
        self.request_history = []
        self.user_sessions = defaultdict(list)
    
    def process_request(self, user_id, request_data):
        """Process a request and potentially cache data"""
        # Simulate processing
        processed_data = {
            'user_id': user_id,
            'timestamp': time.time(),
            'data': request_data.copy(),
            'result': [i ** 2 for i in range(100)]  # Some computation
        }
        
        # Store in various places (potential memory growth)
        self.request_history.append(processed_data)
        self.user_sessions[user_id].append(processed_data)
        
        # Cache based on some logic
        cache_key = f"user_{user_id}_latest"
        self.cache[cache_key] = processed_data
        
        return processed_data

# Run production memory profiling simulation
print("=== Production Memory Profiling Simulation ===")

profiler = MemoryProfiler()
app = ProductionApp()

# Take baseline snapshot
profiler.take_snapshot("Application startup")
print("Baseline snapshot taken")

# Simulate application workload
for iteration in range(3):
    print(f"\n--- Simulation Iteration {iteration + 1} ---")
    
    # Simulate multiple requests
    for i in range(50):
        user_id = f"user_{i % 10}"  # 10 different users
        request_data = {'action': f'action_{i}', 'payload': list(range(20))}
        app.process_request(user_id, request_data)
    
    # Take snapshot after processing
    profiler.take_snapshot(f"After iteration {iteration + 1}")
    
    # Analyze and report
    growth = profiler.analyze_growth()
    if isinstance(growth, dict):
        print(f"Objects grown: {growth['object_growth']:+,}")
        if growth['type_changes']:
            print("Significant changes:", list(growth['type_changes'].keys())[:3])

# Generate final report
print(f"\n{profiler.get_memory_report()}")

# Demonstrate cleanup monitoring
print(f"\n=== Cleanup Monitoring ===")
initial_cache_size = len(app.cache)
app.cache.clear()
app.request_history.clear()
app.user_sessions.clear()

profiler.take_snapshot("After cleanup")
final_growth = profiler.analyze_growth()
print(f"Cache cleaned: {initial_cache_size} items removed")
print(f"Object count change: {final_growth['object_growth']:+,}")

print("Example 5: Production memory profiling workflow completed successfully!")